# Config B — So sánh 4 mô hình hồi quy

## Mục tiêu
So sánh Linear Regression, Random Forest, Gradient Boosting và MLP với bộ tham số chỉnh tay, tham khảo thí nghiệm ngày 12.

- Đặc trưng thời gian, ngày lễ và thời tiết.
- Grid CV theo thời gian trên train để xem kết quả từng bộ tham số.
- Huấn luyện bộ `MODEL_PARAMS` khai báo sẵn và so sánh trên validation bằng MAE, RMSE và R².


## Params

Chỉnh `MODEL_PARAMS` để quyết định bộ tham số dùng cho bảng validation và biểu đồ. Chỉnh `PARAM_GRIDS` để thay đổi các tổ hợp đánh giá CV. `RUN_GRID_CV=True` chạy toàn bộ lưới khi bạn Run All. Đặt `False` khi chỉ muốn chạy bộ tham số chỉnh tay.

Mỗi tổ hợp được đánh giá bằng 3 fold thời gian với train mở rộng dần. Bảng CV được giữ theo thứ tự tổ hợp để bạn đọc và chọn thủ công.

Bộ đề xuất dựa trên ngày 12

- Random Forest tăng từ 200 lên 400 cây, độ sâu từ 12 lên 20, giảm số mẫu lá từ 5 xuống 2 để thử học chi tiết hơn.
- Gradient Boosting tăng từ 200 lên 400 cây, độ sâu từ 3 lên 4, giữ learning rate 0.05 để thử giảm thiếu khớp.
- MLP tăng mạng từ (64, 32) lên (128, 64), tăng alpha lên 0.001 và max_iter lên 1000 để thử cân bằng khả năng học và regularization.
- Linear Regression có ít tham số ảnh hưởng độ linh hoạt, nên lưới kiểm tra fit_intercept.

Đây là giả thuyết trước khi chạy. Đối chiếu MAE ngày 12 lần lượt khoảng 569.61, 240.11, 294.42, 270.22 theo thứ tự bốn mô hình để xác nhận mức thay đổi.


In [ ]:
EXPERIMENT_NAME = "ConfigB_Phuc_13_11_2026"
CONFIG_NAME = "B"
TARGET = "traffic_volume"
RANDOM_STATE = 42
PRIMARY_METRIC = "MAE"
RUN_GRID_CV = True
CV_SPLITS = 3

DATA_PARAMS = {"mask_stuck_temp": False, "complete_holidays": False}

# Bộ đề xuất dùng trực tiếp khi huấn luyện và dự đoán validation.
MODEL_PARAMS = {
    "Linear Regression": {"fit_intercept": True},
    "Random Forest": {
        "n_estimators": 400, "max_depth": 20, "min_samples_leaf": 2,
        "max_features": 1.0, "random_state": RANDOM_STATE, "n_jobs": -1,
    },
    "Gradient Boosting": {
        "n_estimators": 400, "learning_rate": 0.05, "max_depth": 4,
        "min_samples_leaf": 5, "loss": "squared_error", "random_state": RANDOM_STATE,
    },
    "MLP": {
        "hidden_layer_sizes": (128, 64), "activation": "relu", "solver": "adam",
        "alpha": 0.001, "learning_rate_init": 0.001, "batch_size": 128,
        "max_iter": 1000, "random_state": RANDOM_STATE,
    },
}

# Giá trị trong lưới ghi đè đúng khóa tương ứng trong MODEL_PARAMS khi chạy CV.
PARAM_GRIDS = {
    "Linear Regression": {"fit_intercept": [True, False]},
    "Random Forest": {
        "n_estimators": [200, 400], "max_depth": [12, 20],
        "min_samples_leaf": [2, 5],
    },
    "Gradient Boosting": {
        "n_estimators": [200, 400], "max_depth": [3, 4],
        "learning_rate": [0.03, 0.05],
    },
    "MLP": {
        "hidden_layer_sizes": [(64, 32), (128, 64)],
        "alpha": [0.0001, 0.001], "learning_rate_init": [0.0005, 0.001],
    },
}


## Import và tìm thư mục dự án

In [ ]:

import json
import sys
import warnings
from pathlib import Path
from datetime import datetime, timezone, timedelta
from time import perf_counter

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


def find_project_root():
    current = Path.cwd().resolve()

    for folder in (current, *current.parents):
        for candidate in (
            folder,
            folder / "metro_interstate_traffic_volume",
        ):
            if (
                (candidate / "src" / "models.py").is_file()
                and (candidate / "src" / "features.py").is_file()
            ):
                return candidate

    raise FileNotFoundError(
        "Không tìm thấy dự án. Hãy kiểm tra thư mục làm việc của kernel."
    )


PROJECT_ROOT = find_project_root()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import features
from src.data import load_splits, load_hourly_splits
from src.models import make_fixed_model
from src.evaluation import clip_prediction, regression_metrics

DATA_PATH = PROJECT_ROOT / "data" / "Metro_Interstate_Traffic_Volume.csv"

RUN_ID = datetime.now(
    timezone(timedelta(hours=7))
).strftime("%Y%m%d_%H%M%S_%f")

OUTPUT_DIR = (
    PROJECT_ROOT
    / "experiments"
    / "13-11-2026_ConfigB_4Model_Phuc"
    / "outputs"
    / RUN_ID
)

print("Project:", PROJECT_ROOT)
print("Data:", DATA_PATH)
print("Output:", OUTPUT_DIR)

assert DATA_PATH.is_file(), f"Không tìm thấy dữ liệu: {DATA_PATH}"
EXPERIMENT_DIR = PROJECT_ROOT / "experiments" / "13-11-2026_ConfigB_4Model_Phuc"
# Nạp helper bằng đường dẫn cụ thể khi kernel được mở từ thư mục khác.
import importlib.util
spec = importlib.util.spec_from_file_location(
    "config_b_grid_cv", EXPERIMENT_DIR / "config_b_grid_cv.py"
)
grid_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(grid_module)
make_config_b_pipeline = grid_module.make_config_b_pipeline
run_grid_cv = grid_module.run_grid_cv


## Đọc dữ liệu và tạo đúng Config B

In [ ]:
splits = load_splits(
    path=DATA_PATH,
    **DATA_PARAMS,
)

# Lấy danh sách từ features.py, không chép tay 27 tên cột.
FEATURE_COLUMNS = list(features.CONFIG_B)

# Config B không cần tạo history features.
featured = {
    name: features.add_weather_features(
        features.add_time_features(splits[name])
    )
    for name in ("train", "validation")
}

train = featured["train"]
validation = featured["validation"]

assert not train.empty
assert not validation.empty

for name, frame in featured.items():
    assert frame["date_time"].is_monotonic_increasing, name
    assert not frame["date_time"].duplicated().any(), name

    missing = set(FEATURE_COLUMNS + [TARGET]) - set(frame.columns)
    assert not missing, f"{name} thiếu cột: {sorted(missing)}"

assert train["date_time"].max() < validation["date_time"].min()
assert TARGET not in FEATURE_COLUMNS
assert set(FEATURE_COLUMNS).isdisjoint(features.HISTORY_FEATURES)

X_train = train[FEATURE_COLUMNS].copy()
y_train = train[TARGET].copy()

X_val = validation[FEATURE_COLUMNS].copy()
y_val = validation[TARGET].copy()

assert all(pd.api.types.is_numeric_dtype(t) for t in X_train.dtypes)
assert all(pd.api.types.is_numeric_dtype(t) for t in X_val.dtypes)

assert np.isfinite(X_train.to_numpy(dtype=float)).all()
assert np.isfinite(X_val.to_numpy(dtype=float)).all()
assert np.isfinite(y_train.to_numpy(dtype=float)).all()
assert np.isfinite(y_val.to_numpy(dtype=float)).all()

summary = pd.DataFrame([
    {
        "split": name,
        "rows": len(frame),
        "start": frame["date_time"].min(),
        "end": frame["date_time"].max(),
        "n_features": len(FEATURE_COLUMNS),
    }
    for name, frame in featured.items()
])

display(summary)
print("Features:", FEATURE_COLUMNS)
print("X_train:", X_train.shape)
print("X_val:", X_val.shape)

## Gọi file .py và chạy Grid CV

`config_b_grid_cv.py` duyệt lưới bằng `ParameterGrid`, tính điểm trên từng fold thời gian và trả đầy đủ kết quả. Việc điền thời tiết được học riêng từ phần train của từng fold. Bảng validation bên dưới dùng bộ tham số trong `MODEL_PARAMS`.

Lưới mặc định có 26 tổ hợp, tương ứng 78 lần fit CV, sau đó thêm 4 lần fit bộ tham số chỉnh tay. Thời gian chạy phụ thuộc máy và mức hội tụ của MLP.


In [ ]:
from sklearn.model_selection import ParameterGrid

candidate_counts = {name: len(ParameterGrid(grid)) for name, grid in PARAM_GRIDS.items()}
display(pd.DataFrame([
    {"model": name, "n_candidates": count, "n_cv_fits": count * CV_SPLITS}
    for name, count in candidate_counts.items()
]))

if RUN_GRID_CV:
    raw_train = load_hourly_splits(path=DATA_PATH, **DATA_PARAMS)["train"]
    cv_results = run_grid_cv(raw_train, MODEL_PARAMS, PARAM_GRIDS, n_splits=CV_SPLITS)
    display(cv_results)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    cv_export = cv_results.copy()
    cv_export["params"] = cv_export["params"].map(
        lambda params: json.dumps(params, ensure_ascii=False)
    )
    cv_export.to_csv(OUTPUT_DIR / "grid_cv_results.csv", index=False)
    (OUTPUT_DIR / "grid_cv_config.json").write_text(
        json.dumps({"n_splits": CV_SPLITS, "base_params": MODEL_PARAMS,
                    "param_grids": PARAM_GRIDS}, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )


## Huấn luyện bộ tham số chỉnh tay và đánh giá validation

In [ ]:
fitted_models = {}
metric_rows = []
warning_rows = []

predictions = validation[["date_time", TARGET]].copy()

for model_name, params in MODEL_PARAMS.items():
    pipeline = make_config_b_pipeline(model_name, params)

    print(f"Đang chạy: {model_name}")

    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")

        start = perf_counter()
        pipeline.fit(X_train, y_train)
        fit_seconds = perf_counter() - start

        start = perf_counter()
        raw_prediction = pipeline.predict(X_val)
        predict_seconds = perf_counter() - start

    raw_prediction = np.asarray(raw_prediction, dtype=float)
    assert np.isfinite(raw_prediction).all(), model_name

    prediction = clip_prediction(raw_prediction)
    metrics = regression_metrics(y_val, prediction)

    metric_rows.append({
        "config": CONFIG_NAME,
        "model": model_name,
        "split": "validation",
        "n_features": len(FEATURE_COLUMNS),
        "n_train": len(X_train),
        "n_validation": len(X_val),
        "fit_seconds": fit_seconds,
        "predict_seconds": predict_seconds,
        **metrics,
    })

    predictions[f"{model_name}__raw"] = raw_prediction
    predictions[model_name] = prediction
    fitted_models[model_name] = pipeline

    for warning in caught:
        warning_rows.append({
            "model": model_name,
            "category": warning.category.__name__,
            "message": str(warning.message),
        })

results = (
    pd.DataFrame(metric_rows)
    .sort_values(PRIMARY_METRIC)
    .reset_index(drop=True)
)

warning_table = pd.DataFrame(
    warning_rows,
    columns=["model", "category", "message"],
)

display(results.round(4))

if not warning_table.empty:
    display(warning_table)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
results.to_csv(OUTPUT_DIR / "validation_metrics.csv", index=False)
predictions.to_csv(OUTPUT_DIR / "validation_predictions.csv", index=False)
warning_table.to_csv(OUTPUT_DIR / "validation_warnings.csv", index=False)
(OUTPUT_DIR / "manual_params.json").write_text(
    json.dumps(MODEL_PARAMS, ensure_ascii=False, indent=2), encoding="utf-8"
)


## Nhận xét kết quả validation

Sau khi chạy, so sánh MAE, RMSE và R² với kết quả ngày 12. Ghi rõ mô hình cải thiện hoặc giảm chất lượng, cùng bộ tham số đã dùng.


In [ ]:
import pandas as pd
import matplotlib.dates as mdates
import matplotlib.pyplot as plt

HOURS_TO_PLOT = 7 * 24

model_names = [
    "Random Forest",
    "MLP",
    "Gradient Boosting",
    "Linear Regression",
]

prediction_table = predictions.copy()
prediction_table["date_time"] = pd.to_datetime(
    prediction_table["date_time"]
)

plot_index = pd.date_range(
    start=prediction_table["date_time"].min(),
    periods=HOURS_TO_PLOT,
    freq="h",
)

# Giữ NaN ở các giờ thiếu để đường biểu diễn ngắt tại khoảng trống.
plot_data = (
    prediction_table
    .set_index("date_time")
    .reindex(plot_index)
)

fig, axes = plt.subplots(
    len(model_names),
    1,
    figsize=(14, 11),
    sharex=True,
    sharey=True,
    layout="constrained",
    squeeze=False,
)

for ax, name in zip(axes[:, 0], model_names):
    ax.plot(
        plot_data.index,
        plot_data[TARGET],
        color="#202020",
        linewidth=1.5,
        label="Thực tế",
    )

    ax.plot(
        plot_data.index,
        plot_data[name],
        color="tab:orange",
        linestyle="--",
        linewidth=1.2,
        label="Dự đoán",
    )

    ax.set_title(name, loc="left")
    ax.set_ylabel("Xe/giờ")
    ax.grid(alpha=0.25)
    ax.legend(loc="upper right")

axes[-1, 0].xaxis.set_major_locator(
    mdates.HourLocator(byhour=[0, 12])
)
axes[-1, 0].xaxis.set_major_formatter(
    mdates.DateFormatter("%d/%m\n%H:%M")
)
axes[-1, 0].set_xlabel("Ngày và giờ")

fig.suptitle(
    f"Lưu lượng thực tế và dự đoán của bốn mô hình "
    f"— Cấu hình {CONFIG_NAME}"
)
image_dir = OUTPUT_DIR / "img"
image_dir.mkdir(parents=True, exist_ok=True)

image_path = image_dir / f"config_b_4_models_{HOURS_TO_PLOT}h.png"

fig.savefig(
    image_path,
    dpi=300,
    bbox_inches="tight",
    facecolor="white",
)

print("Đã lưu ảnh:", image_path)
plt.show()

## Nhận xét từ hình

Quan sát mức bám sát thực tế của từng mô hình ở giờ cao điểm, thấp điểm và các khoảng sai lệch lớn trong 7 ngày đầu validation. Điền nhận xét sau khi chạy.
